<a href="https://colab.research.google.com/github/vichu2k7-crypto/CyberSecurity-Lab/blob/main/U4e8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def zero_trust_authorize(request, resource_policy):
    """
    Checks:
    1. User authorization
    2. MFA
    3. Antivirus status
    4. OS patch status
    """

    reasons = []

    allowed_users = resource_policy.get(
        request["resource"],
        set()
    )

    # Check whether user is authorized
    if request["user"] not in allowed_users:
        reasons.append(
            "user not authorized for this resource"
        )

    # Check MFA
    if not request["mfa_passed"]:
        reasons.append(
            "MFA not completed"
        )

    # Check antivirus
    if not request["device_posture"]["antivirus_enabled"]:
        reasons.append(
            "antivirus disabled"
        )

    # Check OS patch
    if not request["device_posture"]["os_patched"]:
        reasons.append(
            "OS not fully patched"
        )

    return {
        "granted": len(reasons) == 0,
        "reasons": reasons
    }

In [3]:
def test_experiment8():

    policy = {
        "finance_db": {
            "csmith",
            "afinance"
        }
    }

    # Test 1: Healthy request
    healthy_request = {
        "user": "csmith",
        "mfa_passed": True,
        "device_posture": {
            "antivirus_enabled": True,
            "os_patched": True
        },
        "resource": "finance_db",
    }

    assert zero_trust_authorize(
        healthy_request,
        policy
    )["granted"] is True

    # Test 2: Antivirus disabled
    unhealthy_request = dict(
        healthy_request,
        device_posture={
            "antivirus_enabled": False,
            "os_patched": True
        }
    )

    result2 = zero_trust_authorize(
        unhealthy_request,
        policy
    )

    assert result2["granted"] is False
    assert "antivirus disabled" in result2["reasons"]

    # Test 3: Unauthorized user
    unauthorized_request = dict(
        healthy_request,
        user="attacker99"
    )

    assert zero_trust_authorize(
        unauthorized_request,
        policy
    )["granted"] is False

    print("All test cases passed.")


test_experiment8()

All test cases passed.
